In [1]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)

2.12.1+cu130
0.27.1+cu132


In [5]:
# GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [7]:
!nvidia-smi

Thu Jul 23 17:35:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   46C    P8              2W /  115W |       5MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

1) Get data

In [12]:
import os
import zipfile

from pathlib import Path

import requests

# setting uop data path
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi" #food 101 daatser


In [13]:
# setting up dir path

train_dir = image_path / "train"
test_dir = image_path / "test"

train_dir, test_dir

(PosixPath('data/pizza_steak_sushi/train'),
 PosixPath('data/pizza_steak_sushi/test'))

2. Creating datasets and dataLoaders

In [14]:
from going_modular import data_setup


### 2.1 creating a transform for torchvision.models (manual creation)

In [15]:
from torchvision import transforms
manual_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # 1. Reshape all images to 224x224 (though some models may require different sizes)
    transforms.ToTensor(), # 2. Turn image values to between 0 & 1 
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # 3. A mean of [0.485, 0.456, 0.406] (across each colour channel)
                         std=[0.229, 0.224, 0.225]) # 4. A standard deviation of [0.229, 0.224, 0.225] (across each colour channel),
])

In [16]:
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                                                                               test_dir=test_dir,
                                                                               transform=manual_transforms,
                                                                               batch_size=32)
train_dataloader, test_dataloader, class_names

(<torch.utils.data.dataloader.DataLoader at 0x7ffafc2fef80>,
 ['pizza', 'steak', 'sushi'])

### 2.2 Creating a transform for `torchvision.models` (auto creation)

In [2]:
import torchvision
torchvision.__version__

'0.27.1+cu132'

In [4]:
#getting a set of pretrained model weights
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT #best perfomance weight
weights

EfficientNet_B0_Weights.IMAGENET1K_V1

In [17]:
#Get the transforms used to create our pretrained weights
auto_transforms = weights.transforms()
auto_transforms

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [18]:
# Creating dataloaders using automatioc transforms
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                                                                               test_dir=test_dir,
                                                                               transform=auto_transforms,
                                                                               batch_size=32)
train_dataloader, test_dataloader, class_names

(<torch.utils.data.dataloader.DataLoader at 0x7ff9e2611330>,
 ['pizza', 'steak', 'sushi'])

## 3. Getting a pretrained model